# RARO — Treino dos classificadores de voz "Raro gravar" / "Raro parar" (KAGGLE)

Este notebook treina **dois classificadores de frase inteira** em **português (PT-BR)** usando
[`livekit-wakeword`](https://github.com/livekit/livekit-wakeword) (Apache-2.0, baseado no openWakeWord,
com cabeça Conv-Attention). A saída final é **um único `.zip` com 4 arquivos `.onnx`** prontos para o app iOS.

## Por que KAGGLE (e não Google Colab)

A versão anterior rodava no Google Colab grátis, mas o Colab **desconecta no meio do treino**
(limite de sessão / inatividade), perdendo o progresso. O **Kaggle** dá GPU grátis estável
(**até ~9h por sessão, sem cair**), ideal para os dois treinos seguidos.

## Por que DOIS classificadores (Decisão 5a)

O RARO precisa distinguir dois comandos distintos:

- **"raro gravar"** -> inicia a gravação (`raro_gravar.onnx`)
- **"raro parar"** -> para a gravação (`raro_parar.onnx`)

Wake-word puro só detecta a presença de UMA frase, então rodamos a pipeline DUAS vezes (uma por comando).
Os dois estágios de extração de features (`melspectrogram.onnx` + `embedding_model.onnx`) são **genéricos**
e compartilhados entre os dois comandos — eles vêm prontos dentro do pacote pip, não são treinados aqui.

## Tempo estimado

- Instalação de dependências: ~3-6 min
- Treino do comando START (`raro gravar`): ~10-20 min
- Treino do comando STOP (`raro parar`): ~10-20 min
- **Total: ~20-40 min** (varia com `n_samples`, `steps` e a GPU alocada).

## Saída final

Um `raro_onnx_export.zip` (salvo em `/kaggle/working/`) com **exatamente 4 arquivos**:

| Arquivo | Papel |
|---|---|
| `melspectrogram.onnx` | Extração de features (genérico, vem do pacote) |
| `embedding_model.onnx` | Extração de features (genérico, vem do pacote) |
| `raro_gravar.onnx` | Classificador treinado — comando START |
| `raro_parar.onnx` | Classificador treinado — comando STOP |

Esses 4 vão para `apps/mobile/ios/Runner/Resources/` no projeto RARO.


## Passo 0 — PRÉ-REQUISITOS DO KAGGLE (LEIA ANTES DE RODAR)

O Kaggle é diferente do Colab em DUAS coisas que, se você esquecer, fazem o notebook falhar.
Configure as duas no **painel direito** (clique em **"..."** ou no ícone de engrenagem ->
**Session options** / **Notebook options**):

> ### ⚠️ 1. Conta verificada por telefone
> A GPU grátis e a Internet só ficam disponíveis para contas **verificadas por telefone**.
> Vá em **Settings -> Phone Verification** e confirme seu número. Sem isso, as opções de
> **Accelerator** e **Internet** aparecem **bloqueadas (cinza)** e nada abaixo funciona.

> ### ⚠️ 2. Ligar GPU + Internet no painel direito (Session options)
> - **Accelerator** -> selecione **GPU T4 x2** (ou **GPU P100**). O VoxCPM (gerador de voz PT-BR)
>   precisa de GPU; na CPU o treino é inviável.
> - **Internet** -> **On**. Esta é a **pegadinha nº 1 do Kaggle**: por padrão a Internet vem
>   **DESLIGADA**, e aí o `pip install` (Passo 2) e os downloads de assets/VoxCPM (Passo 4)
>   **falham**. Tem que estar **On**.

Depois de ligar os dois, rode as células **na ordem**, de cima para baixo.

> **Onde fica o resultado:** tudo que for salvo em `/kaggle/working/` aparece na aba **Output**
> do notebook ao fim da sessão — é de lá que você baixa o `raro_onnx_export.zip` (ver Passo 7/9).

> **Cota de disco:** a 1ª célula de código redireciona os downloads grandes (ACAV100M ~17GB + caches
> HuggingFace/Torch) para `/kaggle/temp`, que **não conta** na cota de 20GB de `/kaggle/working` —
> evita o erro "No space left on device".


In [ ]:
# Redireciona TODOS os caches/downloads grandes para /kaggle/temp (scratchpad
# que NAO conta na cota de 20GB do /kaggle/working). Tem que rodar ANTES de
# qualquer import de torch/transformers/huggingface_hub.
import os
TEMP = "/kaggle/temp"
os.environ["HF_HOME"]        = f"{TEMP}/hf"
os.environ["HF_HUB_CACHE"]   = f"{TEMP}/hf/hub"
os.environ["XDG_CACHE_HOME"] = f"{TEMP}/.cache"
os.environ["TORCH_HOME"]     = f"{TEMP}/torch"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"  # acelera download grande
for p in [f"{TEMP}/hf", f"{TEMP}/torch", f"{TEMP}/.cache", f"{TEMP}/raro-data"]:
    os.makedirs(p, exist_ok=True)
print("Caches redirecionados para", TEMP)

### Diagnóstico de disco (cota de 20GB)

Confira na saída abaixo que `/kaggle/temp` (ou `/`) tem **>20GB livres**. Se `/kaggle/temp`
aparecer com pouco espaço, os downloads vão para `/tmp` ou `/` — todos fora da cota de
`/kaggle/working`. O ACAV100M (~17GB) precisa desse espaço separado para não estourar a cota.

In [ ]:
# Diagnostico de disco: confirma que /kaggle/temp tem espaco separado dos 20GB
# de /kaggle/working. O ACAV100M precisa de ~17GB livres aqui.
!df -h /kaggle/working /kaggle/temp /tmp / 2>/dev/null

In [ ]:
# Confirma que uma GPU (T4 ou P100) esta alocada.
# Se falhar ou nao listar GPU: a GPU NAO foi ligada no painel direito.
# Va em Session options -> Accelerator -> GPU T4 x2 (ou P100) e rode de novo.
!nvidia-smi


## Passo 1 — Dependências de sistema

Pacotes nativos exigidos pelo `livekit-wakeword` (fonte: doc oficial):
`espeak-ng` (fonemização), `libsndfile1` (I/O de áudio), `ffmpeg`, `sox`, `portaudio19-dev`.

No Kaggle as células rodam como **root**, então `apt-get install` funciona sem `sudo`.


In [ ]:
# Dependencias de sistema do livekit-wakeword (doc oficial). (pode levar ~2min)
# Kaggle roda como root: nao precisa de sudo.
!apt-get update -qq && apt-get install -y espeak-ng libsndfile1 ffmpeg sox portaudio19-dev


## Passo 2 — Instalar o `livekit-wakeword`

Instalamos com os extras **`train,eval,export,voxcpm`**.

> **Precisa de Internet On (Passo 0).** Se o `pip install` travar ou der erro de rede, a Internet
> do notebook está desligada — volte ao painel **Session options -> Internet -> On**.

> **Importante:** o extra **`voxcpm`** é obrigatório porque vamos gerar voz em **português** com o
> backend VoxCPM. A doc oficial do `prod_voxcpm.yaml` é explícita: *"Multilingual wake words require
> `tts_backend: voxcpm`"* e instrui instalar o pacote **com o extra `voxcpm`**
> (`pip install livekit-wakeword[train,eval,export,voxcpm]`). Sem ele, o backend de voz PT-BR não carrega.


In [ ]:
# Instala o livekit-wakeword com os extras de treino/eval/export + VoxCPM (PT-BR). (pode levar ~3-6min)
# Requer Internet On (Passo 0).
!pip install -q "livekit-wakeword[train,eval,export,voxcpm]"
# Confirma a versao instalada:
!pip show livekit-wakeword | grep -i -E "^(Name|Version):"


## Passo 3 — Escrever os 2 YAMLs de configuração

Cada YAML descreve um treino. Campos principais:

- **`model_name`** — vira o nome do diretório de saída (`output/<model_name>/`) e do `.onnx` exportado.
- **`target_phrases`** — a(s) frase(s) positiva(s) a sintetizar e detectar (em PT-BR).
- **`tts_backend: voxcpm`** — gerador de voz multilíngue (PT-BR). Exige o extra `voxcpm` do Passo 2.
- **`n_samples`** — clipes de treino **por classe** (positivos + negativos). Mais = melhor recall, porém mais lento.
- **`model.model_type: conv_attention`** + **`model.model_size: small`** — cabeça Conv-Attention, tamanho enxuto p/ embarcado.
- **`steps`** — passos de treino (fase 1).
- **`target_fp_per_hour`** — alvo de falsos-positivos/hora para calibragem do threshold.

> **Sobre os valores:** usamos o baseline travado pelo plano (`n_samples: 10000`, `model_size: small`,
> `steps: 50000`, `target_fp_per_hour: 0.2`). O `prod_voxcpm.yaml` oficial usa valores maiores
> (`n_samples: 25000`, `model_size: medium`, `steps: 100000`). **Se o recall vier abaixo de 80%** no
> eval (Passo 8), o caminho é **aumentar `n_samples`/`steps`** e/ou subir `model_size` para `medium`
> (o Kaggle aguenta o treino maior dentro do limite de ~9h).

> **`custom_negative_phrases` em PT-BR:** a doc do `prod_voxcpm.yaml` avisa que os negativos adversariais
> automáticos são *enviesados para inglês (CMUdict)*. Por isso adicionamos manualmente frases-isca
> parecidas em português (ex.: só "raro", "câmera", "gravando") para reduzir falso-positivo. Ajuste à vontade.

A célula seguinte cria a pasta `configs/` antes de escrever os arquivos.


In [ ]:
# Cria a pasta de configs antes dos %%writefile abaixo. (working dir = /kaggle/working)
!mkdir -p configs


In [ ]:
%%writefile configs/raro_gravar.yaml
# Comando START: "raro gravar" -> raro_gravar.onnx
model_name: raro_gravar

# Downloads grandes (ACAV100M ~17GB) vao p/ /kaggle/temp (fora da cota de 20GB).
data_dir: /kaggle/temp/raro-data
# Modelos treinados ficam em /kaggle/working p/ persistir no Output.
output_dir: /kaggle/working/output

target_phrases:
  - "raro gravar"

# Voz sintetica multilingue (PT-BR). Exige o extra 'voxcpm' (Passo 2).
tts_backend: voxcpm

# Clipes de treino por classe. 2000 = treino rapido (~1h Kaggle, cabe no limite de 12h da sessao).
n_samples: 2000
# Clipes de validacao por classe (~10-20% de n_samples).
n_samples_val: 400

# Negativos adversariais em PT-BR (os automaticos sao enviesados p/ ingles). Ajustaveis.
custom_negative_phrases:
  - "raro"
  - "raro foto"
  - "camera"
  - "gravando"
  - "caro gravar"
  - "raro parar"
  - "parou"

# Personas de voz do VoxCPM (grade enxuta; a doc usa uma grade maior).
voxcpm_tts:
  voice_design_prompts:
    - "A young adult woman, clear mid-pitch voice, moderate pace, calm tone"
    - "A young adult man, warm baritone, steady pace, friendly tone"
    - "A middle-aged woman, slightly low pitch, measured pace, confident tone"
  cfg_values: [2.0]
  inference_timesteps_list: [10]

model:
  model_type: conv_attention
  model_size: small

steps: 50000
target_fp_per_hour: 0.2


In [ ]:
%%writefile configs/raro_parar.yaml
# Comando STOP: "raro parar" -> raro_parar.onnx
model_name: raro_parar

# Downloads grandes (ACAV100M ~17GB) vao p/ /kaggle/temp (fora da cota de 20GB).
data_dir: /kaggle/temp/raro-data
# Modelos treinados ficam em /kaggle/working p/ persistir no Output.
output_dir: /kaggle/working/output

target_phrases:
  - "raro parar"

# Voz sintetica multilingue (PT-BR). Exige o extra 'voxcpm' (Passo 2).
tts_backend: voxcpm

# Clipes de treino por classe. 2000 = treino rapido (~1h Kaggle, cabe no limite de 12h da sessao).
n_samples: 2000
# Clipes de validacao por classe (~10-20% de n_samples).
n_samples_val: 400

# Negativos adversariais em PT-BR (os automaticos sao enviesados p/ ingles). Ajustaveis.
# Inclui 'raro gravar' como negativo p/ o modelo de PARAR nao confundir com o de GRAVAR.
custom_negative_phrases:
  - "raro"
  - "raro gravar"
  - "camera"
  - "gravando"
  - "caro parar"
  - "raro parou"
  - "vamos parar"

# Personas de voz do VoxCPM (grade enxuta; a doc usa uma grade maior).
voxcpm_tts:
  voice_design_prompts:
    - "A young adult woman, clear mid-pitch voice, moderate pace, calm tone"
    - "A young adult man, warm baritone, steady pace, friendly tone"
    - "A middle-aged woman, slightly low pitch, measured pace, confident tone"
  cfg_values: [2.0]
  inference_timesteps_list: [10]

model:
  model_type: conv_attention
  model_size: small

steps: 50000
target_fp_per_hour: 0.2


## Passo 4 — Treinar o classificador START (`raro gravar`)

Duas etapas, ambas da doc oficial:

1. **`setup --config`** — baixa os assets compartilhados (features, RIRs, ruídos de fundo) e, como
   `tts_backend: voxcpm`, baixa o snapshot do VoxCPM2 do Hugging Face (pode demorar na 1ª vez).
   **Precisa de Internet On.**
2. **`run`** — pipeline completo num comando: **generate + augment + train + export**.

> Esta etapa **demora** (geração de voz + treino), ~10-20min. O Kaggle não cai, mas não feche a aba.


In [ ]:
# 1/2: baixa assets compartilhados + snapshot do VoxCPM2 (HuggingFace). (pode levar ~3-8min na 1a vez)
# OBS: 'setup' usa --config; 'run'/'eval' usam o config POSICIONAL (sem flag). E proposital.
!livekit-wakeword setup --config configs/raro_gravar.yaml


In [ ]:
# 2/2: generate + augment + train + export num comando so. (pode levar ~10-20min)
!livekit-wakeword run configs/raro_gravar.yaml


## Passo 5 — Treinar o classificador STOP (`raro parar`)

Mesma sequência do Passo 4, agora para o comando de parar. Os assets compartilhados já foram
baixados, então o `setup` aqui tende a ser rápido (só confere o que falta).


In [ ]:
# 1/2: confere/baixa assets (ja em cache do Passo 4, tende a ser rapido). (pode levar ~1-3min)
!livekit-wakeword setup --config configs/raro_parar.yaml


In [ ]:
# 2/2: generate + augment + train + export do comando STOP. (pode levar ~10-20min)
!livekit-wakeword run configs/raro_parar.yaml


## Passo 6 — Localizar TODOS os `.onnx` (treinados + genéricos)

Precisamos de **4** arquivos:

- **2 treinados** (os classificadores): ficam em `output/<model_name>/<model_name>.onnx`,
  ou seja `output/raro_gravar/raro_gravar.onnx` e `output/raro_parar/raro_parar.onnx`.
- **2 genéricos** (extração de features): vêm **dentro do pacote pip**. A doc confirma:
  *"Feature extraction models (`melspectrogram.onnx`, `embedding_model.onnx`) are bundled with the package"*.
  No código-fonte do pacote eles ficam em `livekit/wakeword/resources/`.

A célula abaixo faz uma **busca robusta**: lista todo `*.onnx` no diretório de trabalho **e** dentro do
pacote `livekit.wakeword` instalado, mostrando caminho + tamanho. Assim você **vê** onde cada arquivo caiu
(os caminhos podem variar por versão — por isso buscamos em vez de chutar). Use a saída para conferir/ajustar
o Passo 7 se necessário.


In [ ]:
import glob, os

# O pacote e importado como 'livekit.wakeword' (namespace package 'livekit'), nao 'livekit_wakeword'.
from livekit import wakeword as lkww
pkg_dir = os.path.dirname(lkww.__file__)
print("Diretorio do pacote livekit.wakeword:", pkg_dir)
print("Working dir (Kaggle):", os.getcwd())
print()

def listar(titulo, padroes, base="."):
    print(f"=== {titulo} ===")
    achados = []
    for pad in padroes:
        achados += glob.glob(os.path.join(base, pad), recursive=True)
    achados = sorted(set(achados))
    if not achados:
        print("  (nenhum .onnx encontrado aqui)")
    for p in achados:
        kb = os.path.getsize(p) / 1024
        print(f"  {kb:8.1f} KB  {p}")
    print()
    return achados

# (a) classificadores treinados (e quaisquer .onnx) no diretorio de trabalho
listar("ONNX no working dir (treinados)", ["**/*.onnx"], base=".")

# (b) genericos bundlados dentro do pacote pip
listar("ONNX bundlados no pacote (mel + embedding)", ["**/*.onnx"], base=pkg_dir)


## Passo 7 — Empacotar os 4 `.onnx` no zip (em `/kaggle/working/`)

A célula abaixo copia os 4 arquivos para `/kaggle/working/raro_onnx_export/`, renomeando para **exatamente**
`melspectrogram.onnx`, `embedding_model.onnx`, `raro_gravar.onnx`, `raro_parar.onnx`, valida cada um e
gera `/kaggle/working/raro_onnx_export.zip`.

Para cada arquivo, ela também roda `onnx.checker.check_model` e imprime os nomes/shapes de **input/output**,
para você confirmar que os classificadores batem o contrato esperado pela próxima task:
**input `embeddings` `(1, 16, 96)` float32 -> output `score` `(1, 1)`**.

> **Se o Passo 6 mostrou caminhos/nomes diferentes** dos que estão no dicionário `ORIGENS` abaixo,
> ajuste os valores do dicionário com os caminhos reais que apareceram, e rode esta célula de novo.
> Se algum arquivo não for encontrado, a célula falha com uma mensagem clara dizendo qual ajustar.


In [ ]:
import os, glob, shutil
import onnx
from livekit import wakeword as lkww

pkg_dir = os.path.dirname(lkww.__file__)

# Diretorio de saida do Kaggle (aparece na aba Output ao fim da sessao).
WORKING = "/kaggle/working"

# Caminhos canonicos (ajuste aqui se o Passo 6 mostrou algo diferente).
# Os classificadores saem em output/<model_name>/<model_name>.onnx (relativo ao working dir).
# Os genericos ficam em <pacote>/resources/.
ORIGENS = {
    "melspectrogram.onnx":  os.path.join(pkg_dir, "resources", "melspectrogram.onnx"),
    "embedding_model.onnx": os.path.join(pkg_dir, "resources", "embedding_model.onnx"),
    "raro_gravar.onnx":     os.path.join(WORKING, "output", "raro_gravar", "raro_gravar.onnx"),
    "raro_parar.onnx":      os.path.join(WORKING, "output", "raro_parar", "raro_parar.onnx"),
}

def resolver(nome, caminho_esperado):
    """Usa o caminho canonico; se nao existir, tenta achar por nome via glob e avisa."""
    if os.path.exists(caminho_esperado):
        return caminho_esperado
    candidatos = sorted(set(
        glob.glob(os.path.join(WORKING, "**", nome), recursive=True) +
        glob.glob(os.path.join(".", "**", nome), recursive=True) +
        glob.glob(os.path.join(pkg_dir, "**", nome), recursive=True)
    ))
    if candidatos:
        print(f"[aviso] '{caminho_esperado}' nao existe; usando candidato encontrado: {candidatos[0]}")
        return candidatos[0]
    raise FileNotFoundError(
        f"NAO ENCONTREI '{nome}'. Esperado em: {caminho_esperado}. "
        f"Veja a saida do Passo 6 e ajuste o caminho de '{nome}' no dicionario ORIGENS desta celula."
    )

DEST = os.path.join(WORKING, "raro_onnx_export")
if os.path.exists(DEST):
    shutil.rmtree(DEST)
os.makedirs(DEST, exist_ok=True)

for nome_final, caminho in ORIGENS.items():
    origem = resolver(nome_final, caminho)
    destino = os.path.join(DEST, nome_final)
    shutil.copyfile(origem, destino)
    assert os.path.exists(destino), f"Falha ao copiar {nome_final} (ajuste o caminho do Passo 6)."
    # Valida o ONNX e imprime o contrato de I/O.
    m = onnx.load(destino)
    onnx.checker.check_model(m)
    def fmt(vis):
        out = []
        for v in vis:
            dims = []
            for d in v.type.tensor_type.shape.dim:
                dims.append(str(d.dim_value) if d.dim_value else (d.dim_param or "?"))
            out.append(f"{v.name} [{', '.join(dims)}]")
        return "; ".join(out) if out else "(nenhum)"
    kb = os.path.getsize(destino) / 1024
    print(f"OK  {nome_final}  ({kb:.1f} KB)")
    print(f"      input : {fmt(m.graph.input)}")
    print(f"      output: {fmt(m.graph.output)}")

# Confere que sao exatamente os 4 esperados.
presentes = sorted(os.listdir(DEST))
esperados = sorted(ORIGENS.keys())
assert presentes == esperados, f"Esperado {esperados}, encontrado {presentes}"

zip_path = shutil.make_archive(os.path.join(WORKING, "raro_onnx_export"), "zip", DEST)
print()
print("Zip gerado:", zip_path, f"({os.path.getsize(zip_path)/1024:.1f} KB)")
print("Conteudo  :", presentes)


In [ ]:
# No Kaggle o download NAO usa o helper do Colab; e pela aba Output do notebook.
# Este FileLink cria um link clicavel para o zip salvo em /kaggle/working/.
from IPython.display import FileLink
FileLink("raro_onnx_export.zip")


> **Como baixar no Kaggle:** o `raro_onnx_export.zip` está em `/kaggle/working/` e aparece na aba
> **Output** (ou **Data -> Output**) do notebook ao fim da sessão. Clique no arquivo e use o botão
> de **download** da UI do Kaggle. O `FileLink` acima também serve como atalho clicável dentro do notebook.
> (Não usamos o helper de download do Colab — ele não existe no Kaggle.)


## Passo 8 — (opcional) Eval offline de recall

Mede recall/falsos-positivos no conjunto de validação sintético, sem device. Forma oficial
(config **posicional**, sem `--config`):

```
!livekit-wakeword eval configs/raro_gravar.yaml
!livekit-wakeword eval configs/raro_parar.yaml
```

> **Gate de aceite do projeto: recall > 80%.** Este eval offline (voz sintética) é só um indicador
> precoce — o gate **oficial** é medido no iPhone 12 físico, com voz real, na Task 7.
> **Se o recall offline já vier abaixo de 80%**, pare e avise o time: o plano B é aumentar
> `n_samples`/`steps`, subir `model_size` para `medium`, ou complementar com **gravações reais**
> de voz PT-BR.


In [ ]:
# Eval offline (voz sintetica). Indicador precoce; o gate real e no device (Task 7). (pode levar ~2-5min)
!livekit-wakeword eval configs/raro_gravar.yaml
!livekit-wakeword eval configs/raro_parar.yaml


## Passo 9 — O que fazer com o `.zip`

1. **Baixe** o `raro_onnx_export.zip` pela aba **Output** do notebook no Kaggle (Passo 7).
2. **Descompacte** o zip.
3. Copie os **4 arquivos `.onnx`** para a pasta do projeto RARO:

   ```
   apps/mobile/ios/Runner/Resources/
     melspectrogram.onnx
     embedding_model.onnx
     raro_gravar.onnx
     raro_parar.onnx
   ```
4. **Avise o time** que os 4 modelos estão no lugar — daí seguimos para a **Task 3** (carregar os
   3 estágios no `WakeWordDetector` Swift e implementar a inferência).

Anote também, para as notas de treino (`docs/superpowers/notes/raro-model-training.md`): o **recall**
do Passo 8, o **tamanho** de cada `.onnx` (saída do Passo 7), o **`n_samples`** usado e a **data** do treino.
